<a href="https://colab.research.google.com/github/NITHIN-aiml69/python-project-sem-3/blob/main/MLP/linear_0.4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import pandas as pd

# --- 1. Load data ---
df1 = pd.read_csv('Tuesday-WorkingHours.pcap_ISCX.csv', low_memory=True)
df2 = pd.read_csv('Wednesday-workingHours.pcap_ISCX.csv', low_memory=True)
df3 = pd.read_csv('Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', low_memory=True)

dataset = pd.concat([df1, df2, df3], ignore_index=True)
dataset.columns = dataset.columns.str.strip()

X = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1]

X = X.apply(pd.to_numeric, errors='coerce')
X.replace([np.inf, -np.inf], np.nan, inplace=True)

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
X = imputer.fit_transform(X)

from sklearn.preprocessing import LabelEncoder
labelencoder_y = LabelEncoder()
y = labelencoder_y.fit_transform(y)

# --- 2. SUBSAMPLE (this was missing -> caused the RAM crash) ---
# Kernel PCA needs an n x n matrix in memory, so the full ~1.3M rows
# is impossible. This keeps 15,000 rows with the same class balance.
from sklearn.model_selection import train_test_split
X, _, y, _ = train_test_split(X, y, train_size=15000, random_state=0, stratify=y)

# --- 3. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,        # change to 0.2 or 0.6 for other runs
    random_state=0,
    stratify=y
)

# --- 4. Kernel PCA ---
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(n_components=2, kernel='linear', gamma=15)   # swap kernel/gamma per run
X_train = kpca.fit_transform(X_train)
X_test = kpca.transform(X_test)
# --- 5. Classifier (swap this block for each algorithm) ---
from sklearn.neural_network import MLPClassifier
classifier = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

# --- 6. Metrics ---
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:\n")
print(cm)

print("\nAccuracy : {:.4f}".format(accuracy_score(y_test, y_pred)))
print("\nPrecision : {:.4f}".format(precision_score(y_test, y_pred, average='weighted')))
print("\nRecall : {:.4f}".format(recall_score(y_test, y_pred, average='weighted')))
print("\nF1 Score : {:.4f}".format(f1_score(y_test, y_pred, average='weighted')))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Confusion Matrix:

[[4234    0  165    2    0    0   18    0    0]
 [   7    0    0    0    0    0    0    0    0]
 [ 520    0  889    0    0    0    0    0    0]
 [  10    0   24    0    0    0    0    0    0]
 [  12    0   23    0    0    0    0    0    0]
 [  48    0    0    0    0    0    0    0    0]
 [  22    0    0    0    0    0   13    0    0]
 [   9    0    0    0    0    0    0    0    0]
 [   4    0    0    0    0    0    0    0    0]]

Accuracy : 0.8560

Precision : 0.8329

Recall : 0.8560

F1 Score : 0.8403

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.96      0.91      4419
           1       0.00      0.00      0.00         7
           2       0.81      0.63      0.71      1409
           3       0.00      0.00      0.00        34
           4       0.00      0.00      0.00        35
           5       0.00      0.00      0.00        48
           6       0.42      0.37      0.39        35
           7   

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/m